# JAX Learning Tutorial: From Basics to Advanced

A comprehensive guide to mastering JAX for scientific computing, sampling algorithms, optimization, and deep learning.

## Topics Covered:
1. Introduction & Installation
2. JAX Basics - Arrays and Operations
3. Functional Transformations (grad, jit, vmap)
4. Linear Algebra Operations
5. Random Number Generation (critical for sampling)
6. Automatic Differentiation (autodiff)
7. Optimization Algorithms
8. Neural Networks with Flax
9. Advanced: Functional Programming Patterns
10. Practical: Building a Simple Sampler

## Section 1: Introduction & Installation

JAX is a Python library for high-performance numerical computing with composable function transformations:
- **grad**: Automatic differentiation
- **jit**: Just-in-time compilation
- **vmap**: Vectorization
- **pmap**: Parallel mapping

Key differences from NumPy:
- JAX arrays are immutable
- JAX operations are functional (no side effects)
- Designed for GPU/TPU acceleration
- JIT compilation for performance

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2,3"
import jax
import jax.numpy as jnp
import jax.random as random
import optax
import matplotlib.pyplot as plt

In [2]:
from jax import grad, vmap, pmap, jit

In [3]:
import jax
import jax.numpy as jnp

x = jnp.ones((1000, 1000))
print(jax.devices())
print(jnp.dot(x, x))

[CudaDevice(id=0), CudaDevice(id=1), CudaDevice(id=2)]
[[1000. 1000. 1000. ... 1000. 1000. 1000.]
 [1000. 1000. 1000. ... 1000. 1000. 1000.]
 [1000. 1000. 1000. ... 1000. 1000. 1000.]
 ...
 [1000. 1000. 1000. ... 1000. 1000. 1000.]
 [1000. 1000. 1000. ... 1000. 1000. 1000.]
 [1000. 1000. 1000. ... 1000. 1000. 1000.]]


In [18]:
import jaxlib
print(jaxlib.version.__version__)

0.4.20


In [16]:
print(f"JAX Version: {jax.__version__}")
print(f"Available Devices : {jax.devices()}")
print(f"Default Backend: {jax.default_backend()}")

JAX Version: 0.4.20
Available Devices : [CpuDevice(id=0)]
Default Backend: cpu


In [14]:
from jax.lib import xla_bridge
print(xla_bridge.get_backend().platform)
print(jax.devices())

cpu
[CpuDevice(id=0)]


In [10]:
import jaxlib
print(jaxlib.version.__version__)

0.4.20


In [7]:
import jax
print(jax.devices())

[CpuDevice(id=0)]


In [8]:
from jax.lib import xla_bridge
print(xla_bridge.get_backend().platform)

cpu


In [1]:
# Installation (run in terminal if not already installed):
# pip install jax jaxlib flax optax

# Import essential libraries
import jax
import jax.numpy as jnp
from jax import grad, jit, vmap, pmap
from jax import random
import numpy as np
import matplotlib.pyplot as plt

print(f"JAX version: {jax.__version__}")
print(f"Available devices: {jax.devices()}")
print(f"Default backend: {jax.default_backend()}")

JAX version: 0.4.38
Available devices: [CpuDevice(id=0)]
Default backend: cpu


## Section 2: JAX Basics - Arrays and Operations

### 2.1 Creating JAX Arrays

JAX arrays use `DeviceArray` type and support GPU/TPU acceleration. They are immutable.
Key difference: JAX arrays are functional - operations don't modify originals but return new arrays.

In [9]:
x = jnp.array([1.0, 2.0, 3.0])

In [8]:
x = jnp.array([1.0, 2.0, 3.0])
print(f"JAX array: {x}", x.T)

JAX array: [1. 2. 3.] [1. 2. 3.]


In [9]:
x = jnp.array([1,2,3])
print("Datatype of this jax array:", x.dtype)

Datatype of this jax array: int32


In [12]:
# 2.1 Creating JAX Arrays
x = jnp.array([1.0, 2.0, 3.0, 4.0])
print(f"Array: {x}")
print(f"Type: {type(x)}")
print(f"Shape: {x.shape}")
print(f"Dtype: {x.dtype}")

# Creating arrays with different methods
zeros = jnp.zeros((3, 4))
ones = jnp.ones(5)
arange = jnp.arange(0, 10, 2)
linspace = jnp.linspace(0, 1, 5)
eye = jnp.eye(3)

print(f"\nZeros shape: {zeros.shape}")
print(f"Eye matrix:\n{eye}")

# Most NumPy operations work with jnp
result = jnp.sin(x)
print(f"\nsin(x): {result}")

Array: [1. 2. 3. 4.]
Type: <class 'jaxlib.xla_extension.ArrayImpl'>
Shape: (4,)
Dtype: float32

Zeros shape: (3, 4)
Eye matrix:
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

sin(x): [ 0.84147096  0.9092974   0.14112    -0.7568025 ]


### 2.2 Immutability and Functional Updates

JAX arrays are IMMUTABLE. You can't modify them in-place.

In [7]:
print("At index 0", x[0])

At index 0 1


In [13]:
# 2.2 Immutability - JAX arrays cannot be modified in-place
x = jnp.array([1.0, 2.0, 3.0])

# DON'T TRY THIS (raises error):
# x[0] = 10.0  # TypeError: 'DeviceArray' object does not support item assignment

# Instead, create a new array using .at[] syntax
x_new = x.at[0].set(10.0)
print(f"Original x: {x}")
print(f"Updated x_new: {x_new}")

# Other useful update operations
x_add = x.at[1].add(5.0)  # Add 5 to index 1
x_mul = x.at[2].multiply(3.0)  # Multiply index 2 by 3
print(f"After add: {x_add}")
print(f"After multiply: {x_mul}")

# Batch updates
indices = jnp.array([0, 2])
values = jnp.array([100.0, 300.0])
x_batch = x.at[indices].set(values)
print(f"Batch update: {x_batch}")

Original x: [1. 2. 3.]
Updated x_new: [10.  2.  3.]
After add: [1. 7. 3.]
After multiply: [1. 2. 9.]
Batch update: [100.   2. 300.]


In [ ]:
class velocity:
    def __init__(self, v):
        self.v = v

    def update(self, new_v):
        # Return a new instance with updated velocity
        return velocity(new_v)

## Section 3: Automatic Differentiation with `grad`

`grad` computes gradients of functions. It's one of JAX's most powerful features.

Key points:
- Derivatives are computed symbolically before execution
- Can differentiate w.r.t. any argument
- Can compose multiple derivatives (2nd order, 3rd order, etc.)
- Perfect for optimization and sampling algorithms

In [19]:
def func_2(x):
    return jnp.sin(x) + jnp.cos(x) + jnp.exp(x)

In [20]:
%time
grad_df2 = grad(func_2)
grad_df2(0.0)

CPU times: user 31 µs, sys: 0 ns, total: 31 µs
Wall time: 61 µs


Array(2., dtype=float32, weak_type=True)

In [21]:
def func_3(x):
    return x**3 + x**2 + x + 1

In [22]:
x = 4.0
f_value = func_3(x)
print(f"f(x) value is ,{f_value}")

grad_f = grad(func_3)
df_dx = grad_f(x)
print(f"df_dx at x= {x} is {df_dx}")

f(x) value is ,85.0
df_dx at x= 4.0 is 57.0


In [ ]:
# 3.1 Basic Gradients

# Simple function
def f(x):
    return x**2 + 2*x + 1

x = 3.0
f_value = f(x)
print(f"f({x}) = {f_value}")

# Compute gradient (derivative)
grad_f = grad(f)  # Returns function that computes gradient
df_dx = grad_f(x)
print(f"df/dx at x={x}: {df_dx}")

# Analytical: d/dx(x^2 + 2x + 1) = 2x + 2 = 2*3 + 2 = 8 ✓



In [18]:
# 3.2 Gradients of multivariate functions
def rosenbrock(x):
    """Rosenbrock function for 2D optimization"""
    return (1 - x[0])**2 + 100*(x[1] - x[0]**2)**2

x_vec = jnp.array([0.0, 0.0])
grad_rosenbrock = grad(rosenbrock)
gradient = grad_rosenbrock(x_vec)
print(f"\nGradient of Rosenbrock at {x_vec}: {gradient}")


Gradient of Rosenbrock at [0. 0.]: [-2.  0.]


In [23]:
def new_func(x):
    return jnp.exp(x) + jnp.dot(x,x)

In [27]:
def func_4(x):
    return x[0]**1 + x[1]**2 + x[2]**3

In [29]:
x_vec = jnp.array([2.0,3.0,4.0, 5.0])
new_func(x_vec)
print(f"New Function value is {new_func(x_vec)}")

New Function value is [ 61.389057  74.08554  108.59815  202.41316 ]


In [30]:
grad_new_func = grad(func_4)
gradient_new_func = grad_new_func(x_vec)
print(f"\nGradient of new_func at {x_vec}: {gradient_new_func}")


Gradient of new_func at [2. 3. 4. 5.]: [ 1.  6. 48.  0.]


In [25]:
# 3.3 Higher-order derivatives (Hessian, 2nd derivatives)
def simple_f(x):
    return jnp.sum(x**3)

x = jnp.array([1.0, 2.0])

# First derivative
grad_f = grad(simple_f)
print(f"First derivative at {x}: {grad_f(x)}")

# Second derivative (Hessian diagonal)
hess_diag = grad(lambda x: jnp.sum(grad_f(x)))
print(f"Hessian diagonal at {x}: {hess_diag(x)}")

# 3.4 Gradients w.r.t. specific arguments
def multi_arg_func(x, y):
    return jnp.sum(x**2 + 2*x*y + y**3)

x = jnp.array([1.0, 2.0])
y = jnp.array([3.0, 4.0])

grad_wrt_x = grad(multi_arg_func, argnums=0)  # gradient w.r.t. x (1st arg)
grad_wrt_y = grad(multi_arg_func, argnums=1)  # gradient w.r.t. y (2nd arg)

print(f"\n∂f/∂x: {grad_wrt_x(x, y)}")
print(f"∂f/∂y: {grad_wrt_y(x, y)}")

# Both simultaneously
grad_both = grad(multi_arg_func, argnums=(0, 1))
print(f"Both gradients: {grad_both(x, y)}")

First derivative at [1. 2.]: [ 3. 12.]
Hessian diagonal at [1. 2.]: [ 6. 12.]

∂f/∂x: [ 8. 12.]
∂f/∂y: [29. 52.]
Both gradients: (Array([ 8., 12.], dtype=float32), Array([29., 52.], dtype=float32))


## Section 4: JIT Compilation with `jit`

`jit` (Just-In-Time) compilation traces your function and compiles it to run efficiently on GPU/TPU.

Performance impact:
- First call: slower (compilation overhead)
- Subsequent calls: much faster (10-100x speedup)
- Essential for tight loops in optimization and sampling

In [31]:
# 4.1 Basic JIT Example
import time

def slow_func(x):
    """Function without JIT"""
    total = 0.0
    for i in range(1000):
        total = total + jnp.sin(x[i % len(x)])
    return total

def fast_func(x):
    """Function with JIT"""
    return jnp.sum(jnp.sin(x))  

# Compile JIT version
fast_func_jit = jit(fast_func)

x = jnp.ones(100)

# Warm up (compilation happens here)
_ = fast_func_jit(x)

# Time comparison
n_iter = 1000
start = time.time()
for _ in range(n_iter):
    _ = fast_func(x)
time_normal = time.time() - start

start = time.time()
for _ in range(n_iter):
    _ = fast_func_jit(x)
time_jit = time.time() - start

print(f"Time without JIT: {time_normal:.4f}s")
print(f"Time with JIT: {time_jit:.4f}s")
print(f"Speedup: {time_normal / time_jit:.2f}x")

# 4.2 JIT with Grad
@jit
def loss_func(x):
    return jnp.sum((x - 3.0)**2)

grad_jit = jit(grad(loss_func))
x = jnp.array([1.0, 2.0, 3.0])
gradient = grad_jit(x)
print(f"\nJIT compiled gradient: {gradient}")

Time without JIT: 0.0514s
Time with JIT: 0.0045s
Speedup: 11.49x

JIT compiled gradient: [-4. -2.  0.]


In [34]:
import jax
import jax.numpy as jnp

def f(x):
    return jnp.sum(x**2)  # Maps tensor to scalar

x = jnp.array([[1.0, 2.0], [3.0, 4.0]])
grad = jax.grad(f)(x)  # Gradient w.r.t. each element of x
print(grad)

# Jacobian for tensor-to-tensor
def g(x):
    return x**2  # Maps tensor to tensor

jacobian = jax.jacfwd(g)(x)
# print(jacobian)

[[2. 4.]
 [6. 8.]]


In [32]:
import jax
import jax.numpy as jnp

def f(x):
    return jnp.array([x[0]**2, x[1]**3])

x = jnp.array([2.0, 3.0])
jacobian = jax.jacfwd(f)(x)  # Forward-mode Jacobian
print(jacobian)

[[ 4.  0.]
 [ 0. 27.]]


## Section 5: Vectorization with `vmap`

`vmap` (vectorized map) automatically vectorizes functions over a batch dimension.

Advantages:
- Write code for a single example
- Get batch processing automatically
- No manual loops
- Works with GPU/TPU efficiently
- Critical for MCMC sampling (batch chains)

In [35]:
def single_sample_log_prob(x, mean):
    """Compute log probability for single sample"""
    return - 0.5 * jnp.sum((x- mean)**2)

In [36]:
batch_log_prob = vmap(single_sample_log_prob, in_axes = (0, None))

In [40]:
batch_x = jnp.array([
    [1.0, 2.0, 3.0],
    [1.5, 2.5, 3.5],
    [2.0, 3.0, 4.0],
    [6.0, 7.0, 8.0]
])


print(f"Type of Batch_x is {type(batch_x)}")
print(f"Dimension of Batch x is {batch_x.shape}")

Type of Batch_x is <class 'jaxlib.xla_extension.ArrayImpl'>
Dimension of Batch x is (4, 3)


In [43]:
mean = jnp.array([2.0, 3.0, 4.0])

In [44]:
log_probs = batch_log_prob(batch_x, mean)
print(f"Batch Log Probs Shape: {log_probs.shape}")
print(f"Batch Log Probs: {log_probs}")

Batch Log Probs Shape: (4,)
Batch Log Probs: [ -1.5    -0.375  -0.    -24.   ]


In [ ]:
# 5.1 Basic vmap Example
def single_sample_log_prob(x, mean):
    """Compute log probability for a single sample"""
    return -0.5 * jnp.sum((x - mean)**2)

# Vectorize over batch of samples (first axis)
batch_log_prob = vmap(single_sample_log_prob, in_axes=(0, None))

# Create batch of samples
batch_x = jnp.array([
    [1.0, 2.0, 3.0],
    [1.5, 2.5, 3.5],
    [2.0, 3.0, 4.0]
])
mean = jnp.array([2.0, 3.0, 4.0])

# Run vmap
log_probs = batch_log_prob(batch_x, mean)
print(f"Batch log probs shape: {log_probs.shape}")
print(f"Batch log probs: {log_probs}")

# 5.2 Multiple batch dimensions
def loss_single(x, y):
    return jnp.sum((x - y)**2)

# Vectorize over 2D array of x values and 2D array of y values
batch_x_2d = jnp.ones((5, 3, 2))  # 5 batches, 3 samples, 2 features
batch_y_2d = jnp.ones((5, 3, 2)) * 2.0

batch_loss = vmap(vmap(loss_single))  # nested vmap
losses = batch_loss(batch_x_2d, batch_y_2d)
print(f"\nBatch loss shape: {losses.shape}")
print(f"Batch loss:\n{losses}")

In [45]:
# 5.2 Multiple batch dimensions
def loss_single(x, y):
    return jnp.sum((x - y)**2)

# Vectorize over 2D array of x values and 2D array of y values
batch_x_2d = jnp.ones((5, 3, 2))  # 5 batches, 3 samples, 2 features
batch_y_2d = jnp.ones((5, 3, 2)) * 2.0

batch_loss = vmap(vmap(loss_single))  # nested vmap
losses = batch_loss(batch_x_2d, batch_y_2d)
print(f"\nBatch loss shape: {losses.shape}")
print(f"Batch loss:\n{losses}")


Batch loss shape: (5, 3)
Batch loss:
[[2. 2. 2.]
 [2. 2. 2.]
 [2. 2. 2.]
 [2. 2. 2.]
 [2. 2. 2.]]


## Section 6: Random Number Generation (Critical for Sampling!)

JAX's random number generation uses functional/pure approach - no global state!

Key differences from NumPy:
- Need explicit PRNGKey
- Split keys for independent streams
- Fully deterministic and reproducible
- Essential for sampling algorithms

In [50]:
key = random.PRNGKey(13)
print(f"The Shape of Random key is {key.shape}")
print(f"The Random Key is {key}")

The Shape of Random key is (2,)
The Random Key is [ 0 13]


In [51]:
subkey, key = random.split(key)
x = random.normal(subkey, shape = (3,3))
print(f"The random value distributed as normal of x is {x}")
print(f"The shape of x is {x.shape}")

The random value distributed as normal of x is [[-1.0204768   2.089773    0.49899593]
 [-0.67688906 -1.3185438   1.4023792 ]
 [-1.3998923  -0.19012913  0.3163287 ]]
The shape of x is (3, 3)


In [52]:
key = random.PRNGKey(42)
print(f"The new key is {key}")

The new key is [ 0 42]


In [73]:
key = random.PRNGKey(43)

In [74]:
subkey1 , subkey2, subkey3, subkey4 = random.split(key, 4)


In [75]:
key_dict = {"Sub Key 1": subkey1 , "Sub Key 2": subkey2, "Sub Key 3": subkey3, "Sub Key 4": subkey4}

In [76]:
print(key_dict)

{'Sub Key 1': Array([1759566864, 2309405404], dtype=uint32), 'Sub Key 2': Array([4004327650,  925032842], dtype=uint32), 'Sub Key 3': Array([ 476000355, 1514632593], dtype=uint32), 'Sub Key 4': Array([3442943684,  514437323], dtype=uint32)}


In [77]:
print(f"\n Gaussian : {random.normal(subkey1 , shape = (3,))}")
print(f"\n Uniform: {random.uniform(subkey2, shape = (4,))}")
print(f"\n Exponential: {random.exponential(subkey3, shape = (5,))}")


 Gaussian : [ 1.3429555  -0.45232967  0.5004717 ]

 Uniform: [0.19910443 0.48836935 0.80475616 0.9614465 ]

 Exponential: [2.3832092  0.44004887 0.6690892  2.0444093  0.31919813]


In [ ]:
# 6.1 Creating and splitting keys
key = random.PRNGKey(0)
print(f"PRNG Key: {key}")
print(f"Key shape: {key.shape}")

# Generate random numbers
subkey, key = random.split(key)  # Split for independent random stream
x = random.normal(subkey, shape=(3, 3))
print(f"\nNormal random samples:\n{x}")

# Multiple splits for multiple operations
key = random.PRNGKey(42)
subkey1, subkey2, subkey3, key = random.split(key, 4)

print(f"\nGaussian: {random.normal(subkey1, shape=(2,))}")
print(f"Uniform: {random.uniform(subkey2, shape=(2,))}")
print(f"Exponential: {random.exponential(subkey3, shape=(2,))}")

# 6.2 Reproducibility - same seed gives same results
key1 = random.PRNGKey(0)
sample1 = random.normal(key1, shape=(3,))

key2 = random.PRNGKey(0)
sample2 = random.normal(key2, shape=(3,))

print(f"\nSample 1: {sample1}")
print(f"Sample 2: {sample2}")
print(f"Are they equal? {jnp.allclose(sample1, sample2)}")

# 6.3 Sampling from distributions (for MCMC proposals)
key = random.PRNGKey(0)
subkey, key = random.split(key)

# Multivariate normal
mean = jnp.array([0.0, 0.0])
cov = jnp.eye(2)
sample = random.multivariate_normal(subkey, mean, cov)
print(f"\nMultivariate normal sample: {sample}")

## Section 7: Linear Algebra Operations

JAX provides efficient linear algebra via `jax.numpy.linalg`

Essential for:
- Matrix operations in sampling
- Covariance matrix computations
- Solving linear systems
- Matrix decompositions

In [89]:


A = jnp.array([[1.0, 2.0, 3.0], [2.0, 3.0, 4.0], [3.0, 5.0, 8.0]])
B = jnp.array([[2.0, 5.0, 7.0], [3.0, 5.0, 8.0], [2.0, 3.0, 4.0]])

In [81]:
X = jnp.array((3,4))

In [82]:
print(X)

[3 4]


In [84]:
C = jnp.dot(A,B.T)

In [87]:
from jax.numpy import linalg

In [91]:
import time
start = time.time()
det_A = linalg.det(A)
end = time.time() - start
det_B = linalg.det(B)
print(f"The Determinant of A is {det_A}")
print(f"The Determinant of B is {det_B}")
print(f"The CPU time for one determinant operation is {end}")


The Determinant of A is -1.0
The Determinant of B is 5.0
The CPU time for one determinant operation is 0.0007760524749755859


In [94]:
I = A @ linalg.inv(A)
print(f"The Identity matrix is {I}")

The Identity matrix is [[ 9.9999976e-01  0.0000000e+00  0.0000000e+00]
 [-4.7683716e-07  1.0000000e+00  0.0000000e+00]
 [-1.9073486e-06  0.0000000e+00  1.0000000e+00]]


In [93]:
print(C.shape)
print(type(C))
print(C)

(2, 2)
<class 'jaxlib.xla_extension.ArrayImpl'>
[[33. 37.]
 [47. 53.]]


In [95]:
A = jnp.array([[1.0, 2.0], [3.0, 4.0]])
B = jnp.array([[2.0, 0.0], [1.0, 2.0]])

# Matrix multiplication
C = jnp.dot(A, B)  # or A @ B
print(f"A @ B =\n{C}")

# Determinant
det_A = linalg.det(A)
print(f"\ndet(A) = {det_A}")

# Matrix inverse
A_inv = linalg.inv(A)
print(f"A^-1 =\n{A_inv}")

# Verify A @ A^-1 = I
print(f"A @ A^-1 =\n{A @ A_inv}")


A @ B =
[[ 4.  4.]
 [10.  8.]]

det(A) = -2.0
A^-1 =
[[-2.0000002   1.0000001 ]
 [ 1.5000001  -0.50000006]]
A @ A^-1 =
[[ 1.0000000e+00  0.0000000e+00]
 [-4.7683716e-07  1.0000002e+00]]


In [98]:
L = linalg.cholesky(A)
print(f"\nCholesky L =\n{L}")


Cholesky L =
[[nan  0.]
 [nan nan]]


In [97]:
D = jnp.array([[1.0, 2.0], [3.0, 4.0]])
decomp = linalg.cholesky(D)
print(f"The cholesky decomposition of D is {decomp}")

The cholesky decomposition of D is [[nan  0.]
 [nan nan]]


In [99]:
# 7.2 Matrix decompositions (critical for sampling!)
A = jnp.array([[3.0, 1.0], [1.0, 2.0]])  # Symmetric positive definite

# Cholesky decomposition (for sampling from Gaussians)
L = linalg.cholesky(A)
print(f"\nCholesky L =\n{L}")
print(f"L @ L^T =\n{L @ L.T}")  # Should equal A


Cholesky L =
[[1.7320508  0.        ]
 [0.57735026 1.2909945 ]]
L @ L^T =
[[3.         0.99999994]
 [0.99999994 2.0000002 ]]


In [100]:
# Eigendecomposition
eigenvalues, eigenvectors = linalg.eigh(A)
print(f"\nEigenvalues: {eigenvalues}")
print(f"Eigenvectors:\n{eigenvectors}")


Eigenvalues: [1.3819661 3.618034 ]
Eigenvectors:
[[ 0.52573115 -0.85065085]
 [-0.85065085 -0.52573115]]


In [101]:
# 7.3 Solving linear systems
A = jnp.array([[3.0, 1.0], [1.0, 2.0]])
b = jnp.array([5.0, 3.0])

# Solve Ax = b
x = linalg.solve(A, b)
print(f"\nSolution to Ax = b: {x}")
print(f"Verify A @ x = b: {A @ x}")


Solution to Ax = b: [1.4 0.8]
Verify A @ x = b: [5. 3.]


In [ ]:
# 7.1 Basic matrix operations


A = jnp.array([[1.0, 2.0], [3.0, 4.0]])
B = jnp.array([[2.0, 0.0], [1.0, 2.0]])

# Matrix multiplication
C = jnp.dot(A, B)  # or A @ B
print(f"A @ B =\n{C}")

# Determinant
det_A = linalg.det(A)
print(f"\ndet(A) = {det_A}")

# Matrix inverse
A_inv = linalg.inv(A)
print(f"A^-1 =\n{A_inv}")

# Verify A @ A^-1 = I
print(f"A @ A^-1 =\n{A @ A_inv}")

# 7.2 Matrix decompositions (critical for sampling!)
A = jnp.array([[3.0, 1.0], [1.0, 2.0]])  # Symmetric positive definite

# Cholesky decomposition (for sampling from Gaussians)
L = linalg.cholesky(A)
print(f"\nCholesky L =\n{L}")
print(f"L @ L^T =\n{L @ L.T}")  # Should equal A

# Eigendecomposition
eigenvalues, eigenvectors = linalg.eigh(A)
print(f"\nEigenvalues: {eigenvalues}")
print(f"Eigenvectors:\n{eigenvectors}")

# 7.3 Solving linear systems
A = jnp.array([[3.0, 1.0], [1.0, 2.0]])
b = jnp.array([5.0, 3.0])

# Solve Ax = b
x = linalg.solve(A, b)
print(f"\nSolution to Ax = b: {x}")
print(f"Verify A @ x = b: {A @ x}")

## Section 8: Optimization with Optax

Optax is a gradient-based optimization library that works seamlessly with JAX.

Key optimizers:
- SGD, Adam, RMSprop, etc.
- Learning rate schedules
- Gradient clipping, weight decay
- Perfect for training neural networks and fitting models

In [102]:
try:
    import optax
except:
    print("Installing optax")
    import subprocess
    subprocess.check_call(["pip", "install", "optax"])
    import optax

def loss_fn(param):
    x, y = param
    return (x - 4.0)**2 + (y + 2.0)**2

In [105]:
params = jnp.array([2.0, 3.0])
# Init optiimizer 
optimizer = optax.adam(learning_rate = 0.03)
opt_state = optimizer.init(params)

print("Optimization Iterations")
num_steps = 10
for step in range(num_steps):
    loss, grads = jax.value_and_grad(loss_fn)(params)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    print(f"Step: {step}, loss : {loss:.4f}, params: {params}, opt_state: {opt_state}, updates: {updates}")

print(f"\n Final Optimum value is : {params}")


loss_fn(params)

Optimization Iterations
Step: 0, loss : 29.0000, params: [2.0299997 2.9700003], opt_state: (ScaleByAdamState(count=Array(1, dtype=int32), mu=Array([-0.4,  1. ], dtype=float32), nu=Array([0.016, 0.1  ], dtype=float32)), EmptyState()), updates: [ 0.0299998 -0.0299998]
Step: 1, loss : 28.5818, params: [2.0599868 2.9400053], opt_state: (ScaleByAdamState(count=Array(2, dtype=int32), mu=Array([-0.75400007,  1.894     ], dtype=float32), nu=Array([0.0315076 , 0.19870362], dtype=float32)), EmptyState()), updates: [ 0.02998702 -0.02999485]
Step: 2, loss : 28.1673, params: [2.0899527 2.9100184], opt_state: (ScaleByAdamState(count=Array(3, dtype=int32), mu=Array([-1.0666027,  2.692601 ], dtype=float32), nu=Array([0.0465307, 0.2961195], dtype=float32)), EmptyState()), updates: [ 0.02996586 -0.02998695]
Step: 3, loss : 27.7566, params: [2.119888 2.880043], opt_state: (ScaleByAdamState(count=Array(4, dtype=int32), mu=Array([-1.3419518,  3.4053445], dtype=float32), nu=Array([0.0610773 , 0.39225653], d

Array(24.991333, dtype=float32)

In [ ]:
# 8.1 Basic optimization loop with optax
try:
    import optax
except:
    print("Installing optax...")
    import subprocess
    subprocess.check_call(["pip", "install", "optax"])
    import optax

# Define loss function
def loss_fn(params):
    x, y = params
    return (x - 3.0)**2 + (y + 2.0)**2  # Minimum at (3, -2)

# Initialize parameters and optimizer
params = jnp.array([0.0, 0.0])
optimizer = optax.adam(learning_rate=0.1)
opt_state = optimizer.init(params)

# Training loop
print("Optimization iterations:")
for step in range(10):
    loss, grads = jax.value_and_grad(loss_fn)(params)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    print(f"Step {step}: loss={loss:.4f}, params={params}")

print(f"\nFinal optimum: {params} (target: [3, -2])")

# 8.2 Training with gradient descent  
@jit
def update_step(params, opt_state):
    loss, grads = jax.value_and_grad(loss_fn)(params)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

# Reset
params = jnp.array([0.0, 0.0])
opt_state = optimizer.init(params)
optimizer = optax.sgd(learning_rate=0.1)
opt_state = optimizer.init(params)

print("\nJIT-compiled optimization:")
for step in range(5):
    params, opt_state, loss = update_step(params, opt_state)
    print(f"Step {step}: loss={loss:.4f}")

In [107]:
import jax
import jax.numpy as jnp

# Example scalar parameter
params = {"w": jnp.array(0.0)}

def loss_fn(params, x, y):
    pred = params["w"] * x
    return jnp.mean((pred - y) ** 2)

x = jnp.array([1., 2., 3., 4.])
y = jnp.array([2., 4., 6., 8.])

lr = 0.1

for step in range(10):
    loss, grads = jax.value_and_grad(loss_fn)(params, x, y)

    # Gradient flow checks
    grad_norm = jnp.sqrt(sum(jnp.sum(g**2) for g in jax.tree_util.tree_leaves(grads)))
    has_nan = jnp.any(jnp.array([jnp.any(jnp.isnan(g)) for g in jax.tree_util.tree_leaves(grads)]))

    # SGD update
    params = jax.tree_util.tree_map(lambda p, g: p - lr * g, params, grads)

    print(f"step={step:02d} loss={loss:.6f} grad_norm={grad_norm:.6f} w={params['w']:.6f} nan_grad={bool(has_nan)}")


step=00 loss=30.000000 grad_norm=30.000000 w=3.000000 nan_grad=False
step=01 loss=7.500000 grad_norm=15.000000 w=1.500000 nan_grad=False
step=02 loss=1.875000 grad_norm=7.500000 w=2.250000 nan_grad=False
step=03 loss=0.468750 grad_norm=3.750000 w=1.875000 nan_grad=False
step=04 loss=0.117188 grad_norm=1.875000 w=2.062500 nan_grad=False
step=05 loss=0.029297 grad_norm=0.937500 w=1.968750 nan_grad=False
step=06 loss=0.007324 grad_norm=0.468750 w=2.015625 nan_grad=False
step=07 loss=0.001831 grad_norm=0.234375 w=1.992188 nan_grad=False
step=08 loss=0.000458 grad_norm=0.117188 w=2.003906 nan_grad=False
step=09 loss=0.000114 grad_norm=0.058594 w=1.998047 nan_grad=False


In [106]:
@jit
def update_step(params, opt_state):
    loss, grads = jax.value_and_grad(loss_fn)(params)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params,updates)
    return params, opt_state, loss

params = jnp.array([0.0, 0.0])
opt_state = optimizer.init(params)
optimizer = optax.sgd(learning_rate = 0.01)
opt_state = optimizer.init(params)

print("\n JIT-Compiled Optimization:")
for step in range(10):
    params, opt_state, loss = update_step(params, opt_state)
    print(f"step: {step}, loss: {loss:.4f}, opt_state: {opt_state}")


 JIT-Compiled Optimization:
step: 0, loss: 20.0000, opt_state: (EmptyState(), EmptyState())
step: 1, loss: 19.2080, opt_state: (EmptyState(), EmptyState())
step: 2, loss: 18.4474, opt_state: (EmptyState(), EmptyState())
step: 3, loss: 17.7168, opt_state: (EmptyState(), EmptyState())
step: 4, loss: 17.0153, opt_state: (EmptyState(), EmptyState())
step: 5, loss: 16.3415, opt_state: (EmptyState(), EmptyState())
step: 6, loss: 15.6943, opt_state: (EmptyState(), EmptyState())
step: 7, loss: 15.0728, opt_state: (EmptyState(), EmptyState())
step: 8, loss: 14.4760, opt_state: (EmptyState(), EmptyState())
step: 9, loss: 13.9027, opt_state: (EmptyState(), EmptyState())


## Section 9: Neural Networks with Flax

Flax is a neural network library built on JAX.

Key concepts:
- Modules define network architecture
- `apply` method runs the network
- Parameters are immutable (functional style)
- Works perfectly with JAX transformations

In [1]:
import jax
import jax.random as random
from jax import vmap, pmap, grad, jit
import jax.numpy as jnp


In [2]:
import flax.linen as nn

In [3]:
# Define a simple MLP
class MLP(nn.Module):
    features: list  # List of layer sizes
    
    def setup(self):
        self.layers = [nn.Dense(feat) for feat in self.features]
    
    def __call__(self, x):
        for i, layer in enumerate(self.layers[:-1]):
            x = layer(x)
            x = nn.relu(x)
        x = self.layers[-1](x)
        return x

In [4]:
# Create model and initialize parameters
model = MLP(features=[64, 32, 10])
key = random.PRNGKey(0)
x_dummy = jnp.ones((1, 20))  # Batch of 1, input dimension 20

In [5]:
print(model)

MLP(
    # attributes
    features = [64, 32, 10]
)


In [7]:
print(x_dummy.shape)

(1, 20)


In [8]:

# Initialize parameters
params = model.init(key, x_dummy)
print(f"Model parameters initialized")
print(f"Parameter keys: {params.keys()}")

Model parameters initialized
Parameter keys: dict_keys(['params'])


In [9]:
# Forward pass
output = model.apply(params, x_dummy)
print(f"\nOutput shape: {output.shape}")
print(f"Output sample:\n{output}")

# 9.2 Training a simple network
def loss_function(params, x, y):
    predictions = model.apply(params, x)
    return jnp.mean((predictions - y)**2)


Output shape: (1, 10)
Output sample:
[[ 0.35168678  0.96790725 -0.75876486 -0.15521106 -0.77747655  0.10825817
  -0.03827693 -0.70994514  0.41927752  0.3727334 ]]


In [10]:
def loss_function(params, x, y):
    predictions  = model.apply(params,x)
    return jnp.mean((predictions - y)**2)

In [11]:
key = random.PRNGKey(42)
subkey , key = random.split(key)
x_train = random.normal(subkey, shape = (32, 20)) # The Shape Contains tuple (batch_size, feature dim)
y_train = random.normal(key, shape = (32, 10)) # The Shape of labels contains tuple (batch_size, label dim (one-hot of 10 classes))

In [15]:
import optax
optimizer = optax.adam(learning_rate = 0.01)
opt_state = optimizer.init(params)

@jit
def train_step(params, opt_state, x, y):
    loss, grads = jax.value_and_grad(loss_function)(params, x,y)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

num_steps = 10
for step in range(num_steps):
    params, opt_state, loss = train_step(params,opt_state, x_train,y_train)
    print(f"\n Training Step {step} : , Training Loss: {loss:.4f} ")



 Training Step 0 : , Training Loss: 0.3657 

 Training Step 1 : , Training Loss: 0.3028 

 Training Step 2 : , Training Loss: 0.2485 

 Training Step 3 : , Training Loss: 0.2054 

 Training Step 4 : , Training Loss: 0.1677 

 Training Step 5 : , Training Loss: 0.1388 

 Training Step 6 : , Training Loss: 0.1168 

 Training Step 7 : , Training Loss: 0.0989 

 Training Step 8 : , Training Loss: 0.0856 

 Training Step 9 : , Training Loss: 0.0753 


In [ ]:
# Create dummy training data
key = random.PRNGKey(1)
subkey, key = random.split(key)
x_train = random.normal(subkey, shape=(32, 20))
y_train = random.normal(key, shape=(32, 10))

# Optimizer
optimizer = optax.adam(learning_rate=0.001)
opt_state = optimizer.init(params)

# Training step
@jit
def train_step(params, opt_state, x, y):
    loss, grads = jax.value_and_grad(loss_function)(params, x, y)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

# Train for a few steps
print("\nTraining:")
for step in range(5):
    params, opt_state, loss = train_step(params, opt_state, x_train, y_train)
    print(f"Step {step}: loss={loss:.4f}")

In [16]:
class MLP(nn.Module):
    layer_sizes: list # List of layer sizes

    def setup(self):
        self. layers = [nn.Dense(layer) for layer in self.layer_sizes]
    
    def __call__(self, x):
        for i, layer in enumerate(self.layers[:-1]):
            x = layer(x)
            x = nn.relu(x)
        x = self.layers[-1](x)
        

In [27]:
model = MLP(layer_sizes = [64, 32, 10])

In [18]:
print(model)

MLP(
    # attributes
    layer_sizes = [64, 32, 10]
)


In [23]:
key = random.PRNGKey(2)
x_dummy = jnp.ones((1,20))

In [24]:
print(x_dummy)

[[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]]


In [25]:
params = model.init(key, x_dummy)
print(f"THe model parameters are {params}")
print(f"The Shape of the parameter matrix is {params.keys}")

THe model parameters are {'params': {'layers_0': {'kernel': Array([[-0.02307114,  0.06739965, -0.10890657, ..., -0.28631127,
        -0.1097751 , -0.19335487],
       [ 0.37582976, -0.27075058,  0.01964948, ..., -0.0842685 ,
         0.20252076,  0.24024394],
       [ 0.2527848 , -0.03323412, -0.29091898, ...,  0.34578997,
         0.00180086, -0.23131493],
       ...,
       [ 0.38496804, -0.04593944, -0.04874598, ..., -0.26017553,
         0.15312004,  0.05520256],
       [-0.11085696,  0.21352963, -0.22295463, ..., -0.25651306,
         0.46293688, -0.16934817],
       [-0.11308956, -0.22916634,  0.17784823, ..., -0.38300455,
        -0.27424702,  0.23120743]], dtype=float32), 'bias': Array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32)}, '

In [29]:
# Define a simple MLP
class MLP(nn.Module):
    features: list  # List of layer sizes
    
    def setup(self):
        self.layers = [nn.Dense(feat) for feat in self.features]
    
    def __call__(self, x):
        for i, layer in enumerate(self.layers[:-1]):
            x = layer(x)
            x = nn.relu(x)
        x = self.layers[-1](x)
        return x

# Create model and initialize parameters
model = MLP(features=[64, 32, 10])
key = random.PRNGKey(0)
x_dummy = jnp.ones((1, 20))  # Batch of 1, input dimension 20

# Initialize parameters
params = model.init(key, x_dummy)
print(f"Model parameters initialized")
print(f"Parameter keys: {params.keys()}")

Model parameters initialized
Parameter keys: dict_keys(['params'])


In [30]:
# Forward pass
output = model.apply(params, x_dummy)
print(f"\nOutput shape: {output.shape}")
print(f"Output sample:\n{output}")


Output shape: (1, 10)
Output sample:
[[ 0.35168678  0.96790725 -0.75876486 -0.15521106 -0.77747655  0.10825817
  -0.03827693 -0.70994514  0.41927752  0.3727334 ]]


In [36]:
def loss_func(params, x, y):
    preds = model.apply(params, x)
    return jnp.mean((preds - y)**2)

In [32]:

# Create dummy training data
key = random.PRNGKey(1)
subkey, key = random.split(key)
x_train = random.normal(subkey, shape=(32, 20))
y_train = random.normal(key, shape=(32, 10))

In [45]:
import optax
optimizer = optax.adam(learning_rate= 0.003)
opt_state = optimizer.init(params)

@jit
def train_step( params, opt_state, x ,y ):
    loss, grads = jax.value_and_grad(loss_function)(params, x, y)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss



In [46]:
num_steps = 20

# Training Loop:
for step in range(num_steps):
    params, opt_state, loss = train_step( params, opt_state, x_train, y_train)
    print(f"\n Train [Step : {step}]: Training Loss : {loss:.4f} ")


 Train [Step : 0]: Training Loss : 1.0348 

 Train [Step : 1]: Training Loss : 0.9567 

 Train [Step : 2]: Training Loss : 0.8928 

 Train [Step : 3]: Training Loss : 0.8405 

 Train [Step : 4]: Training Loss : 0.7969 

 Train [Step : 5]: Training Loss : 0.7589 

 Train [Step : 6]: Training Loss : 0.7245 

 Train [Step : 7]: Training Loss : 0.6929 

 Train [Step : 8]: Training Loss : 0.6639 

 Train [Step : 9]: Training Loss : 0.6359 

 Train [Step : 10]: Training Loss : 0.6088 

 Train [Step : 11]: Training Loss : 0.5828 

 Train [Step : 12]: Training Loss : 0.5582 

 Train [Step : 13]: Training Loss : 0.5347 

 Train [Step : 14]: Training Loss : 0.5123 

 Train [Step : 15]: Training Loss : 0.4914 

 Train [Step : 16]: Training Loss : 0.4716 

 Train [Step : 17]: Training Loss : 0.4528 

 Train [Step : 18]: Training Loss : 0.4343 

 Train [Step : 19]: Training Loss : 0.4163 


In [38]:
# 9.1 Simple neural network with Flax
try:
    from flax import linen as nn
except:
    print("Installing flax...")
    import subprocess
    subprocess.check_call(["pip", "install", "flax"])
    from flax import linen as nn

# Define a simple MLP
class MLP(nn.Module):
    features: list  # List of layer sizes
    
    def setup(self):
        self.layers = [nn.Dense(feat) for feat in self.features]
    
    def __call__(self, x):
        for i, layer in enumerate(self.layers[:-1]):
            x = layer(x)
            x = nn.relu(x)
        x = self.layers[-1](x)
        return x

# Create model and initialize parameters
model = MLP(features=[64, 32, 10])
key = random.PRNGKey(0)
x_dummy = jnp.ones((1, 20))  # Batch of 1, input dimension 20

# Initialize parameters
params = model.init(key, x_dummy)
print(f"Model parameters initialized")
print(f"Parameter keys: {params.keys()}")

# Forward pass
output = model.apply(params, x_dummy)
print(f"\nOutput shape: {output.shape}")
print(f"Output sample:\n{output}")

# 9.2 Training a simple network
def loss_function(params, x, y):
    predictions = model.apply(params, x)
    return jnp.mean((predictions - y)**2)

# Create dummy training data
key = random.PRNGKey(1)
subkey, key = random.split(key)
x_train = random.normal(subkey, shape=(32, 20))
y_train = random.normal(key, shape=(32, 10))

# Optimizer
optimizer = optax.adam(learning_rate=0.001)
opt_state = optimizer.init(params)

# Training step
@jit
def train_step(params, opt_state, x, y):
    loss, grads = jax.value_and_grad(loss_function)(params, x, y)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

# Train for a few steps
print("\nTraining:")
for step in range(5):
    params, opt_state, loss = train_step(params, opt_state, x_train, y_train)
    print(f"Step {step}: loss={loss:.4f}")

Model parameters initialized
Parameter keys: dict_keys(['params'])

Output shape: (1, 10)
Output sample:
[[ 0.35168678  0.96790725 -0.75876486 -0.15521106 -0.77747655  0.10825817
  -0.03827693 -0.70994514  0.41927752  0.3727334 ]]

Training:
Step 0: loss=1.2011
Step 1: loss=1.1627
Step 2: loss=1.1272
Step 3: loss=1.0942
Step 4: loss=1.0635


### Task: Write a Validation Splitting (Random) using `jax.random`

### Task: Write Adam and SGD Optimizer using `jax.numpy`

## Section 10: Advanced - Composing Transformations

One of JAX's superpowers: composing `jit`, `grad`, `vmap` together!

This is essential for efficient sampling algorithms:

In [47]:
def loss_fn(params, x, y):
    preds = jnp.dot(params, x)
    return jnp.mean((preds - y)**2)

grad_loss_jit = jit(grad(loss_fn, argnums = 0))
x = jnp.array([1.0, 2.0, 3.0])
y = jnp.array([1.5])

gradient = grad_loss_jit(params, x, jnp.array([2.0]))
print(f"JIT Compiled Gradient: {gradient}")

def single_loss(params, x, y):
    pred = jnp.dot(params,x)
    return (pred - y)**2

batch_grad = vmap(grad(single_loss), in_axes = (None, 0, 0))

TypeError: dot requires ndarray or scalar arguments, got <class 'dict'> at position 0.

In [ ]:
# 10.1 Composing jit + grad
def loss_fn(params, x, y):
    predictions = jnp.dot(params, x)
    return jnp.mean((predictions - y)**2)

# Compose JIT with grad
grad_loss_jit = jit(grad(loss_fn, argnums=0))

params = jnp.array([1.0, 2.0, 3.0])
x = jnp.array([0.5, 0.4, 0.3])
y = jnp.array([1.5])

# This is compiled!
gradient = grad_loss_jit(params, x, jnp.array([2.0]))
print(f"JIT-compiled gradient: {gradient}")

# 10.2 vmap + grad for batch gradients
def single_loss(params, x, y):
    pred = jnp.dot(params, x)
    return (pred - y)**2

# Vectorize gradient computation over batch
batch_grad = vmap(grad(single_loss), in_axes=(None, 0, 0))

params = jnp.array([1.0, 2.0])
batch_x = jnp.array([
    [0.1, 0.2],
    [0.3, 0.4],
    [0.5, 0.6]
])
batch_y = jnp.array([1.0, 2.0, 3.0])

gradients = batch_grad(params, batch_x, batch_y)
print(f"\nBatch gradients shape: {gradients.shape}")
print(f"Batch gradients:\n{gradients}")

# 10.3 Practical: Metropolis-Hastings step with all transformations
def log_prob(x):
    """Log probability function (unnormalized)"""
    return -0.5 * jnp.sum(x**2)

def metropolis_hastings_step(key, current_x, log_prob_fn, proposal_std=0.1):
    """Single MCMC step with jit-compiled acceptance"""
    # Propose new state
    subkey, key = random.split(key)
    proposed_x = current_x + proposal_std * random.normal(subkey, shape=current_x.shape)
    
    # Compute log acceptance ratio
    log_alpha = log_prob_fn(proposed_x) - log_prob_fn(current_x)
    
    # Accept or reject
    subkey, key = random.split(key)
    accept = jnp.log(random.uniform(subkey)) < log_alpha
    
    # Update state
    new_x = jnp.where(accept, proposed_x, current_x)
    return new_x, key, accept

# Compile for speed
mh_step_jit = jit(metropolis_hastings_step)

# Run a short chain
key = random.PRNGKey(0)
x = jnp.array([0.0, 0.0])

print("\nMCMC chain (first 10 steps):")
for i in range(10):
    x, key, accept = mh_step_jit(key, x, log_prob)
    print(f"Step {i}: x={x}, accept={accept}")

# Vectorize over multiple chains
batch_mh_step = vmap(
    lambda key, x: metropolis_hastings_step(key, x, log_prob),
    in_axes=(0, 0)
)

# Multiple chains in parallel
key = random.PRNGKey(0)
keys = random.split(key, 4)
batch_x = jnp.zeros((4, 2))  # 4 chains, 2D state

batch_x_new, keys_new, accepts = batch_mh_step(keys, batch_x)
print(f"\nParallel chains shape: {batch_x_new.shape}")

## Section 11: Best Practices for Sampling Algorithms

### Key Takeaways for Building Samplers with JAX:

1. **Pure Functional Style**
   - Don't mutate state, return new values
   - Use `.at[]` for array updates
   - Pass PRNGKey explicitly

2. **Use JIT Compilation**
   - Wrap tight loops with `@jit`
   - Dramatic speedups (10-100x)
   - Compile once, run many times

3. **Vectorization for Multiple Chains**
   - Use `vmap` for parallel chains
   - Efficient GPU/TPU utilization
   - Natural batching for ensemble methods

4. **Gradients for Efficient Sampling**
   - Use `grad` for Hamiltonian methods
   - Langevin dynamics
   - Variational inference

5. **Random Number Generation**
   - Split keys for independent streams
   - Deterministic and reproducible
   - Critical for MCMC debugging

6. **Memory Efficiency**
   - Stack computations when possible
   - Use in-place updates with `.at[]`
   - Be careful with large batch dimensions

## Section 12: Practical Example - Bayesian Linear Regression with HMC

Combines everything: gradients, linear algebra, random numbers, and optimization.

In [ ]:
# 12.1 Generate synthetic data
key = random.PRNGKey(42)
n_samples = 50
n_features = 3

# True parameters
true_w = jnp.array([1.5, -2.0, 0.5])
true_sigma = 0.1

# Generate data
subkey, key = random.split(key)
X = random.normal(subkey, shape=(n_samples, n_features))

subkey, key = random.split(key)
y = X @ true_w + true_sigma * random.normal(subkey, shape=(n_samples,))

print(f"Data shape: X={X.shape}, y={y.shape}")

# 12.2 Bayesian linear regression - posterior over weights
def log_posterior(w, X, y, prior_std=1.0):
    """Log posterior: log p(w|X,y) = log p(y|X,w) + log p(w)"""
    # Likelihood
    predictions = X @ w
    sigma = 0.1
    log_likelihood = -0.5 * jnp.sum(((y - predictions) / sigma)**2)
    
    # Prior
    log_prior = -0.5 * jnp.sum((w / prior_std)**2)
    
    return log_likelihood + log_prior

# 12.3 Gradient-based sampling (simplified Langevin dynamics)
def langevin_step(key, w, X, y, step_size=0.01):
    """One step of Langevin dynamics: w_new = w + 0.5*step_size*grad_log_p + sqrt(step_size)*z"""
    subkey, key = random.split(key)
    
    # Compute gradient of log posterior
    grad_log_p = grad(log_posterior)(w, X, y)
    
    # Langevin update
    z = random.normal(subkey, shape=w.shape)
    w_new = w + 0.5 * step_size * grad_log_p + jnp.sqrt(step_size) * z
    
    return w_new, key

# Compile for speed
langevin_step_jit = jit(langevin_step)

# Run sampling
w_samples = []
w = jnp.zeros(n_features)
key = random.PRNGKey(0)

print("\nLangevin sampling (first 5 iterations):")
for i in range(100):
    w, key = langevin_step_jit(key, w, X, y)
    if i % 20 == 0:
        print(f"Step {i}: w={w}, log_post={log_posterior(w, X, y):.2f}")
    if i >= 50:  # Collect samples after burn-in
        w_samples.append(w)

w_samples = jnp.array(w_samples)

print(f"\nSampled weights mean: {jnp.mean(w_samples, axis=0)}")
print(f"True weights: {true_w}")
print(f"Sampled weights std: {jnp.std(w_samples, axis=0)}")

# 12.4 Visualization of posterior
print("\nPosterior summary:")
for i in range(n_features):
    mean = jnp.mean(w_samples[:, i])
    std = jnp.std(w_samples[:, i])
    print(f"w[{i}]: {mean:.3f} ± {std:.3f}")

## Section 13: Summary & Next Steps

### What We Covered:
✓ JAX basics and array operations  
✓ Automatic differentiation (`grad`)  
✓ JIT compilation (`jit`)  
✓ Vectorization (`vmap`)  
✓ Random number generation (Pure functional RNG)  
✓ Linear algebra operations  
✓ Optimization with Optax  
✓ Neural networks with Flax  
✓ Composing transformations  
✓ Practical sampling example  

### Key JAX Concepts to Remember:
1. **Functional Programming**: Arrays are immutable, functions are pure
2. **Transformations**: Functions that transform functions (`grad`, `jit`, `vmap`)
3. **Composability**: Chain transformations for powerful effects
4. **GPU/TPU Ready**: Automatic device placement, SPMD-friendly design
5. **Deterministic**: Explicit random number handling for reproducibility

### Recommended Next Steps:

**For Sampling Algorithms:**
- Study Hamiltonian Monte Carlo (HMC)
- Learn about No-U-Turn Sampler (NUTS)
- Explore variational inference

**For Deep Learning:**
- Build more complex networks with Flax
- Learn about batch normalization, dropout
- Study attention mechanisms

**For Linear Algebra:**
- Study matrix decompositions in depth
- Learn SVD, QR, Cholesky for different applications
- Understand spectral methods

**For Optimization:**
- Explore learning rate schedules
- Learn about adaptive methods (Adam, RMSprop)
- Study gradient clipping and normalization

### Useful Resources:
- JAX Documentation: https://jax.readthedocs.io/
- JAX GitHub: https://github.com/google/jax
- Flax Documentation: https://flax.readthedocs.io/
- Optax Documentation: https://optax.readthedocs.io/
- Tutorial example: jax.readthedocs.io/en/latest/notebooks/thinking_in_jax.html

## Appendix: Practice Exercises

### Easy:
1. **Matrix Trace**: Write a function to compute the trace of a matrix, then use `grad` to get its gradient w.r.t. input
2. **Batch Normalization**: Compute mean and std for a batch, apply normalization with `vmap`

### Medium:
3. **Custom Optimizer**: Implement momentum SGD using JAX arrays and immutable updates
4. **Sampling from Gaussian Mixture**: Use `vmap` and `random` module to sample from multiple Gaussians in parallel

### Advanced:
5. **Implement Hamiltonian Monte Carlo**: Full HMC with leapfrog integrator and acceptance check
6. **Neural ODE**: Build a simple continuous ODE solver integrated with a neural network
7. **Variational Autoencoder**: Implement VAE with Flax and optimize with Optax

## Section 14: Replacing Python Training Loops with `lax.scan`

`lax.scan` is JAX's loop primitive for repeated computation with carry state.
- Carry: params, optimizer state, RNG, etc.
- `xs`: per-step inputs (or `None`)
- Output: final carry + stacked per-step outputs


In [ ]:
from jax import lax
import time


### 14.1 `lax.scan` Basics


In [ ]:
def scan_add(carry, x):
    carry = carry + x
    y = carry
    return carry, y

xs = jnp.array([1., 2., 3., 4.])
carry_final, ys = lax.scan(scan_add, 0., xs)
print('final carry:', carry_final)
print('outputs:', ys)


### 14.2 Small Training Setup


In [ ]:
class TinyMLP(nn.Module):
    hidden: int = 64
    out_dim: int = 10

    @nn.compact
    def __call__(self, x):
        x = nn.Dense(self.hidden)(x)
        x = nn.relu(x)
        x = nn.Dense(self.out_dim)(x)
        return x

key = random.PRNGKey(0)
B, D_in, D_out = 128, 32, 10
x_batch = random.normal(key, (B, D_in))
key, k = random.split(key)
y_batch = random.normal(k, (B, D_out))

model_scan = TinyMLP(hidden=64, out_dim=D_out)
params_scan = model_scan.init(key, x_batch)
opt_scan = optax.adam(1e-3)
opt_state_scan = opt_scan.init(params_scan)

def loss_scan(params, x, y):
    pred = model_scan.apply(params, x)
    return jnp.mean((pred - y) ** 2)

@jax.jit
def train_step_scan(params, opt_state, x, y):
    loss, grads = jax.value_and_grad(loss_scan)(params, x, y)
    updates, opt_state = opt_scan.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss


### 14.3 Python Loop Version


In [ ]:
@jax.jit
def run_steps_python(params, opt_state, x, y, num_steps):
    losses = []
    for _ in range(num_steps):
        params, opt_state, loss = train_step_scan(params, opt_state, x, y)
        losses.append(loss)
    return params, opt_state, jnp.stack(losses)

num_steps = 50
p_py, o_py, losses_py = run_steps_python(params_scan, opt_state_scan, x_batch, y_batch, num_steps)
losses_py.block_until_ready()
print('python loop final loss:', float(losses_py[-1]))


### 14.4 `lax.scan` Version


In [ ]:
@jax.jit
def run_steps_scan(params, opt_state, x, y, num_steps):
    def body(carry, _):
        p, o = carry
        p, o, loss = train_step_scan(p, o, x, y)
        return (p, o), loss

    (params, opt_state), losses = lax.scan(body, (params, opt_state), xs=None, length=num_steps)
    return params, opt_state, losses

p_sc, o_sc, losses_sc = run_steps_scan(params_scan, opt_state_scan, x_batch, y_batch, num_steps)
losses_sc.block_until_ready()
print('scan final loss:', float(losses_sc[-1]))


### 14.5 Timing and Speed Implications


In [ ]:
def time_fn(fn, *args, reps=10):
    fn(*args)[-1].block_until_ready()  # warmup
    t0 = time.perf_counter()
    for _ in range(reps):
        out = fn(*args)
        out[-1].block_until_ready()
    return (time.perf_counter() - t0) / reps

t_py = time_fn(run_steps_python, params_scan, opt_state_scan, x_batch, y_batch, num_steps)
t_sc = time_fn(run_steps_scan, params_scan, opt_state_scan, x_batch, y_batch, num_steps)

print(f'python loop avg: {t_py*1000:.3f} ms')
print(f'lax.scan avg   : {t_sc*1000:.3f} ms')
print(f'speedup (py/scan): {t_py/t_sc:.3f}x')


### 14.6 Exercise

Refactor your earlier Section 9 training loop with `lax.scan`:
1. Carry `(params, opt_state)`
2. Return stacked per-step losses
3. Keep shapes static for stable compilation
